![DB Academy](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/common/db-academy.png)

# 09L - Deploy a Declarative Automation Bundle (DAB) to Multiple Environments

### Estimated Duration: ~15 minutes

## Overview

In this lab, you'll extend a single-environment bundle (the one you built in **02L - Deploy a Simple DAB**) so that it can deploy the same job to **both** development and production environments with different configurations per target. You'll move the job into its own resource YAML file, define **bundle variables**, and use **target-level overrides** to point each environment at the correct catalog.

## Learning Objectives

By the end of this lab, you will be able to:

1. **Modularize a bundle** by moving a job definition into a separate YAML file under `resources/` and pulling it in via the `include` mapping.
2. **Define and reference bundle variables** so the same configuration adapts to dev vs prod.
3. **Override job parameters at the target level** so dev and prod read from and write to different catalogs.
4. **Validate, deploy, and run** the bundle against both `dev` and `prod` targets using the Databricks CLI.
5. **Verify** the deployed job actually produced the expected tables in each environment.

## Reference Documentation

- **What are bundles?** (intro): [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/)
- **Bundle configuration reference (full YAML key list)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/reference) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/reference) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/reference)
- **Bundle settings (mappings, including `include` and `targets`)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/settings) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/settings) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/settings)
- **Variables and substitutions**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/variables) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/variables) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/variables)
- **Deployment modes (`development` / `production`)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/deployment-modes) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/deployment-modes) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/deployment-modes)
- **`databricks bundle` CLI commands**: [AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)
- **Set a bundle run identity (`run_as`)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/run-as) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/run-as) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/run-as)

## REQUIRED - SELECT A COMPUTE ENVIRONMENT
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select All-Purpose Compute</strong>
  <div style="color:#333;">

This notebook requires **all-purpose compute** (Dedicated). Serverless is not supported for this notebook.

Follow these steps to attach an all-purpose compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your `labuser_USERNAME` cluster.
    - By default, the notebook might use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

⚠️ **NOTE:** If the cluster shows a **terminated** state (red dot in the cluster picker), it needs to be started before you can attach. Click the cluster, then **Start**, and wait a few minutes until you see a green dot.
  </div>
</div>


## REQUIRED - DATA SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Data Setup</strong>
  <div style="color:#333;">

Recall that your environment was set up using the **02 - REQUIRED - Course Setup and Authentication** notebook.

If you end your lab or your lab session times out, your environment will be reset. You will need to rerun the **02 - REQUIRED - Course Setup and Authentication** notebook to recreate the catalogs and refresh your Databricks CLI credentials.

  </div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course.

In [0]:
%run ../Includes/Classroom-Setup-09L

## B. Lab Scenario

You are responsible for deploying Databricks projects in your organization using **Declarative Automation Bundles (DABs)**. 

You configured the project to deploy to a single development environment in **09L - Deploy a Simple DAB**. 

Your next task is to extend the bundle so the **same** job can be deployed to **both** development and production environments with different configurations per target. You'll accomplish this with **bundle variables** and **target-level overrides**.

### Development target requirements

- Read from and write to your **labuser_UNIQUE_ID_1_dev** catalog (development data, ~100 rows).

### Production target requirements

- Read from and write to your **labuser_UNIQUE_ID_3_prod** catalog (production data, ~22,000 rows).

## C. Preview the Development and Production Data

### C1. View the Development Data

1. Preview the **nyctaxi_raw** development data within your **labuser_UNIQUE_ID_1_dev** catalog. 

    Notice that the dev data contains 100 rows.


In [0]:
spark.sql(f'''
          SELECT * 
          FROM {catalog_dev}.default.nyctaxi_raw
          ''').display()

2. View the tables in your **labuser_UNIQUE_ID_1_dev** catalog. 
    
    Notice that the **nyctaxi_bronze** and **nyctaxi_silver** tables do not exist.


In [0]:
spark.sql(f'SHOW TABLES IN {catalog_dev}.default').display()

### C2. View the Production Data


1. Preview the **nyctaxi_raw** production data within your **labuser_UNIQUE_ID_3_prod** catalog. 

    Notice that the production data contains about 22,000 rows.

In [0]:
spark.sql(f'''
          SELECT count(*) AS TotalRows 
          FROM {catalog_prod}.default.nyctaxi_raw
          ''').display()

In [0]:
spark.sql(f'''
          SELECT * 
          FROM {catalog_prod}.default.nyctaxi_raw
          ''').display()

2. View the tables in your **labuser_UNIQUE_ID_3_prod** catalog. Notice that the **nyctaxi_bronze** and **nyctaxi_silver** tables do not exist.

In [0]:
spark.sql(f'SHOW TABLES IN {catalog_prod}.default').display()

## D. Pre-flight Checks

Before starting the lab tasks, run a couple of quick checks to confirm the Databricks CLI is installed and authenticated against your workspace.

### D1. Check the Databricks CLI Version

Run a CLI command to confirm the Databricks CLI version is **v0.298.0**.

In [0]:
%sh
databricks -v

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    DATABRICKS CLI ERROR TROUBLESHOOTING:
  </strong>
  <div style="color:#333;">

  - If you encounter a Databricks CLI authentication error, it means the authentication was not successful. Confirm you ran the notebook using your **all purpose compute**.

  - If you encounter the error below, it means your **databricks.yml** file has syntax issues due to a modification. Even for non-DAB CLI commands, the **databricks.yml** file is still required, as it may contain important authentication details, such as the host and profile, which are utilized by the CLI commands.

![CLI Invalid YAML](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/databricks_cli_error_invalid_yaml.png)
  </div>
</div>



## E. Task 1 - Get Your Lab User Name

You'll need your lab user name in the next task to populate a bundle variable. Run the cell below to print it.

In [0]:
print(my_catalog)

## F. Task 2 - Update the Resource YAML File

In a new tab, open the **./resources/lab09_nyc.job.yml** file and complete the following:

**Step 2.1** - Set the job `name` so it dynamically appends your user name: `name: lab09_dab_${workspace.current_user.userName}`

**Step 2.2** - Under `parameters`, add the `${bundle.target}` substitution as the default for `display_target`
  - This is so the parameter automatically reflects which target the bundle was deployed to.

**HINT:** Variables and substitutions documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/variables) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/variables) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/variables)

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION (Resource YAML)</summary>

```yaml
resources:
  jobs:
    lab09_dab:
      name: lab09_dab_${workspace.current_user.userName}   # <--- append your user name
      tasks:
        - task_key: create_nyc_tables
          notebook_task:
            notebook_path: ../src/our_project_code.sql
            source: WORKSPACE
      parameters:
        - name: display_target
          default: ${bundle.target}                        # <--- bundle.target substitution
```
</details>

## G. Task 3 - Update **databricks.yml**

In the same tab, open the **databricks.yml** file. First explore the bundle and then complete the four sub-steps below.

**What to notice in the existing file:**

- The bundle is named `demo09_lab_bundle`.
- The `include` mapping is **empty** (you'll fix that in 3.1).
- The `variables` mapping defines several variables (you'll set one in 3.2).
- The `targets` mapping has a `dev` and a `prod` target (you'll add a parameter override to each in 3.3 and 3.4).

### Step 3.1 - Add the resource file to `include`

Add **./resources/lab09_nyc.job.yml** to the `include` mapping so the job you edited in Task 2 is pulled into the bundle.
  - **HINT:** `include` mapping documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/settings#include) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/settings#include) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/settings#include)

### Step 3.2 - Set the `user_name` variable

Set the `user_name` variable's value to your lab user name (from Task 1). 
  - The `user_name` variable feeds the `catalog_dev` and `catalog_prod` variables, so getting this right keeps every reference correct.

### Step 3.3 - Override `catalog_name` for the `dev` target

  - Under the `dev` target, add a job parameter named `catalog_name` whose default is `${var.catalog_dev}`.

### Step 3.4 - Override `catalog_name` for the `prod` target

  - Under the `prod` target, add a job parameter named `catalog_name` whose default is `${var.catalog_prod}`.

**Why this works:** the same job runs against the dev catalog or the prod catalog depending on which target you deploy to, no duplication of the job definition required.

**NOTE:** A complete example **databricks.yml** is in the **solutions** folder if you get stuck.

## H. Task 4 - Validate the Bundle

Validate your **databricks.yml** bundle configuration file using the Databricks CLI. Run the cell and confirm validation succeeds. If there is an error, fix the **databricks.yml** file and re-run.

**HINT:** `databricks bundle` CLI commands documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

In [0]:
# <FILL-IN>

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
     Troubleshooting
  </strong>
  <div style="color:#333;">
If you see the following error after validating your bundle, the format of your notebook could be incorrect.

`Error: notebook src/xxx not found`. 

Check the format of your notebook and adjust accordingly. 

  </div>
</div>



##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks bundle validate
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

</details>

## I. Task 5 - Deploy to the `dev` Target

Deploy the bundle to the development environment using the Databricks CLI.

After the cell completes:

- Manually check that the job was created successfully. The job name will be **[dev <user>] lab09_dab_<userName>**.
- Check the **Job parameters** and confirm:
    - `catalog_name` references your `labuser_UNIQUE_ID_1_dev` catalog
    - `display_target` is `dev`

**NOTE:** Deployment will take about a minute to complete.

**HINT:** `databricks bundle` CLI commands documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

In [0]:
# <FILL-IN>

#### Checkpoint - Dev Deployment
![Dev](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/multiple-env-lab/dev-deployment.png)

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks bundle deploy -t dev
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

</details>

## J. Task 6 - Run the `dev` Job

Run the deployed job against the `dev` target.

**NOTE:** This will take 1-2 minutes to complete.

**HINT:** Use the **job key** from the `resources` mapping (your name will differ):

```yaml
resources:
  jobs:
    lab09_dab:    # <--- This is the job key
      name: lab09_dab_${workspace.current_user.userName}
```

In [0]:
# <FILL-IN>

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks bundle run -t dev lab09_dab
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

</details>

## K. Task 7 - Verify the `dev` Tables

After the job completes, run the following cells to confirm:

- Both **nyctaxi_bronze** and **nyctaxi_silver** tables exist in your **labuser_UNIQUE_ID_1_dev** catalog.
- The **nyctaxi_bronze** table contains **100 rows**.

In [0]:
spark.sql(f'SHOW TABLES IN {catalog_dev}.default').display()

In [0]:
check_nyctaxi_bronze_table(user_catalog = catalog_dev, total_count=100)

## L. Task 8 - Deploy to the `prod` Target

Deploy the bundle to the production environment using the Databricks CLI.

After the cell completes:

- Manually check that the job was created successfully. The production job name will be **lab09_dab_<userName>** (no `[dev …]` prefix in `production` mode).
- Check the **Job parameters** and confirm:
    - `catalog_name` references your `labuser_UNIQUE_ID_3_prod` catalog
    - `display_target` is `prod`

**NOTE:** Deployment will take about a minute to complete.

**HINT:** `databricks bundle` CLI commands documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

**NOTE:** In real production, you typically run the job using a service principal. See the **Set a bundle run identity** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/run-as) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/run-as) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/run-as). For this lab, we run the production job as the user.

In [0]:
# <FILL-IN>

#### Checkpoint - Prod Deployment
![Dev](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/multiple-env-lab/prod-deployment.png)


##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks bundle deploy -t prod
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

</details>

## M. Task 9 - Run the `prod` Job

Run the deployed job against the `prod` target.

**NOTE:** This will take 1-2 minutes to complete.

In [0]:
# <FILL-IN>

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks bundle run -t prod lab09_dab
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

</details>

## N. Task 10 - Verify the `prod` Tables

After the job completes, run the following cells to confirm:

- Both **nyctaxi_bronze** and **nyctaxi_silver** tables exist in your **labuser_UNIQUE_ID_3_prod** catalog.
- The **nyctaxi_bronze** table contains **21,932 rows**.

In [0]:
spark.sql(f'SHOW TABLES IN {catalog_prod}.default').display()

In [0]:
check_nyctaxi_bronze_table(user_catalog = catalog_prod, total_count=21932)

## O. Further Reading

This was a simple example of deploying a DAB to multiple environments. As you go further, two areas worth exploring:

- **Other ways to set a variable's value.** 
  - In this lab, you set values inside **databricks.yml**. 
  - You can also pass values via the Databricks CLI, environment variables, or a `.databrickscfg` profile. 
  - **Set a variable's value** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/variables#set-a-variables-value) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/variables#set-a-variables-value) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/variables#set-a-variables-value)

- **Override cluster settings per environment.** 
  - A common production pattern is to use small clusters in dev and larger clusters or Serverless in prod. 
  - **Override cluster settings in bundles** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/cluster-override) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/cluster-override) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/cluster-override)

## Conclusion

Nice work. In this lab you took a single-environment bundle and extended it to deploy the same job to two environments without duplicating the job definition:

1. Moved the job into **./resources/lab09_nyc.job.yml** and pulled it in via the `include` mapping.
2. Set the `user_name` bundle variable so per-target catalog variables resolve correctly.
3. Added a `catalog_name` job parameter override under each of the `dev` and `prod` targets.
4. Validated, deployed, and ran the bundle against both targets with `databricks bundle validate`, `databricks bundle deploy -t <target>`, and `databricks bundle run -t <target> lab09_dab`.
5. Verified each environment's **nyctaxi_bronze** table had the expected row count (100 in dev, 21,932 in prod).

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>